<a href="https://colab.research.google.com/github/INBAATHITHAN/Athithan_21/blob/main/CROWD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics

import cv2
import math
from collections import defaultdict, deque
from ultralytics import YOLO
import numpy as np

# ============================================================
# AUTONOMOUS VISION & BEHAVIOUR UNDERSTANDING
# Scenario: Workplace Safety
#
# Detects people
# Tracks the same person
# Checks movement
# Detects unusual stationary behaviour
# Identifies WHO and WHEN
# ============================================================


# ---------------- SETTINGS ----------------

MODEL = "yolo11n.pt"

# Person class in COCO dataset
PERSON_CLASS = 0

# Person is considered unusual if stationary for 10 seconds
STATIONARY_TIME = 10

# Maximum movement allowed during stationary period
MOVEMENT_THRESHOLD = 35

# Detection confidence
CONFIDENCE = 0.4

# ---------------- LOAD YOLO ----------------

model = YOLO(MODEL)

# ---------------- VIDEO INPUT ----------------

video_path = "/content/crowd.mp4.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video.")
    exit()

# Video information
fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# ---------------- VIDEO OUTPUT ----------------

output_path = "behaviour_result.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

# ============================================================
# TRACKING DATA
# ============================================================

# Stores movement history for every person

person_history = defaultdict(
    lambda: deque(maxlen=int(STATIONARY_TIME * fps))
)

# Stores IDs for which an abnormal event was already detected

reported_events = set()


# ============================================================
# PROCESS VIDEO
# ============================================================

frame_number = 0

events = []

results = model.track(
    source=video_path,
    stream=True,
    persist=True,
    tracker="bytetrack.yaml",
    classes=[PERSON_CLASS],
    conf=CONFIDENCE,
    verbose=False
)


for result in results:

    frame = result.orig_img.copy()

    frame_number += 1

    current_time = frame_number / fps

    # --------------------------------------------------------
    # Check if people were detected
    # --------------------------------------------------------

    if result.boxes is not None:

        if result.boxes.id is not None:

            boxes = result.boxes.xyxy.cpu().numpy()

            ids = result.boxes.id.int().cpu().tolist()

            confidences = result.boxes.conf.cpu().tolist()

            # ------------------------------------------------
            # Process every detected person
            # ------------------------------------------------

            for box, person_id, confidence in zip(
                boxes,
                ids,
                confidences
            ):

                x1, y1, x2, y2 = map(int, box)

                # Calculate center of bounding box

                center_x = int((x1 + x2) / 2)

                center_y = int((y1 + y2) / 2)

                center = (center_x, center_y)

                # Save person's position

                person_history[person_id].append(
                    (frame_number, center)
                )

                # ------------------------------------------------
                # Calculate movement
                # ------------------------------------------------

                history = person_history[person_id]

                movement = 999999

                if len(history) > 2:

                    first_position = history[0][1]

                    last_position = history[-1][1]

                    movement = math.sqrt(
                        (last_position[0] - first_position[0]) ** 2
                        +
                        (last_position[1] - first_position[1]) ** 2
                    )

                # ------------------------------------------------
                # Behaviour classification
                # ------------------------------------------------

                unusual = False

                if len(history) >= int(
                    STATIONARY_TIME * fps * 0.8
                ):

                    if movement < MOVEMENT_THRESHOLD:

                        unusual = True

                # ------------------------------------------------
                # NORMAL BEHAVIOUR
                # ------------------------------------------------

                if not unusual:

                    label = "NORMAL - MOVING"

                    box_color = (0, 255, 0)

                # ------------------------------------------------
                # ABNORMAL BEHAVIOUR
                # ------------------------------------------------

                else:

                    label = "UNUSUAL - STATIONARY"

                    box_color = (0, 0, 255)

                    # Report event only once

                    if person_id not in reported_events:

                        event = (
                            f"Time: {current_time:.1f}s | "
                            f"Person ID: {person_id} | "
                            f"Unusual stationary behaviour"
                        )

                        print(event)

                        events.append(event)

                        reported_events.add(person_id)

                # ------------------------------------------------
                # Draw bounding box
                # ------------------------------------------------

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    box_color,
                    2
                )

                # ------------------------------------------------
                # Display Person ID
                # ------------------------------------------------

                cv2.putText(
                    frame,
                    f"Person ID: {person_id}",
                    (x1, max(y1 - 30, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    box_color,
                    2
                )

                # ------------------------------------------------
                # Display behaviour
                # ------------------------------------------------

                cv2.putText(
                    frame,
                    label,
                    (x1, y2 + 25),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    box_color,
                    2
                )

                # ------------------------------------------------
                # Display confidence
                # ------------------------------------------------

                cv2.putText(
                    frame,
                    f"Confidence: {confidence:.2f}",
                    (x1, y2 + 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255, 255, 255),
                    1
                )

                # ------------------------------------------------
                # Display movement
                # ------------------------------------------------

                if movement < 999999:

                    cv2.putText(
                        frame,
                        f"Movement: {movement:.1f}px",
                        (x1, y2 + 70),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (255, 255, 255),
                        1
                    )

    # ========================================================
    # HEADER
    # ========================================================

    cv2.rectangle(
        frame,
        (0, 0),
        (width, 65),
        (30, 30, 30),
        -1
    )

    cv2.putText(
        frame,
        "AUTONOMOUS VISION & BEHAVIOUR MONITOR",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Time: {current_time:.1f}s",
        (width - 150, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    # ========================================================
    # EVENT WARNING
    # ========================================================

    if len(events) > 0:

        cv2.rectangle(
            frame,
            (10, height - 55),
            (width - 10, height - 10),
            (0, 0, 255),
            2
        )

        cv2.putText(
            frame,
            "WARNING: UNUSUAL BEHAVIOUR DETECTED",
            (25, height - 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 0, 255),
            2
        )

    # ========================================================
    # SHOW VIDEO
    # ========================================================

    # cv2.imshow is not supported in Colab's headless environment
    # cv2.imshow(
    #     "Autonomous Behaviour Understanding",
    #     frame
    # )

    # Save output video

    out.write(frame)

    # Press Q to quit

    if cv2.waitKey(1) & 0xFF == ord("q"):

        break


# ============================================================
# RELEASE
# ============================================================

cap.release()

out.release()

cv2.destroyAllWindows()


# ============================================================
# PRINT FINAL EVENTS
# ============================================================

print("\n========================================")
print("        BEHAVIOUR ANALYSIS REPORT")
print("========================================")

if len(events) == 0:

    print("No unusual behaviour detected.")

else:

    for event in events:

        print(event)

print("========================================")

print(
    f"Annotated video saved as: {output_path}"
)

Time: 16.9s | Person ID: 27 | Unusual stationary behaviour

        BEHAVIOUR ANALYSIS REPORT
Time: 16.9s | Person ID: 27 | Unusual stationary behaviour
Annotated video saved as: behaviour_result.mp4
